# IC₅₀ analysis

This tutorial determines the IC₅₀ of two enzyme inhibitors. Enzyme activity was
measured in percent of the uninhibited control at 11 inhibitor concentrations,
from 0.5 nM to 30 µM. Each inhibitor was tested in three independent
experiments, with technical triplicates at every concentration. The data are in
{download}`ic50.csv <data/ic50.csv>`.

In [ ]:
import matplotlib.pyplot as plt

import bindcurve as bc

## Load the data

The file is in wide format: the three technical replicates are in the columns
`response_1` to `response_3`.

In [ ]:
data = bc.DoseResponseData.from_csv("data/ic50.csv", format="wide")
data.summary()

Each inhibitor has three independent experiments (`N_exp`) and 99 measured
responses (`N_obs`): 11 concentrations, 3 technical replicates and 3
experiments.

## Fit

The logistic model is fitted to each independent experiment. All four
parameters are fitted, since the data cover both plateaus.

In [ ]:
results = bc.fit(data, "ic50")
results.experiments()[
    ["compound_id", "experiment_id", "IC50", "IC50_stderr", "hill_slope", "r_squared", "warnings"]
]

None of the six fits has a quality warning. The IC₅₀ values of the three
independent experiments of each inhibitor differ by more than the standard
error of a single fit, because the standard error does not include the
variation between experiments.

## Check the fits

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), sharey=True)
for ax, compound in zip(axes, data.compounds):
    bc.plot_fits(results, compounds=compound, ax=ax)
    ax.set_title(compound)
    ax.set_xlabel("inhibitor (nM)")
    ax.legend()
axes[0].set_ylabel("activity (% of control)")
plt.show()

The curves follow the replicate means, and both plateaus and the midpoint are
covered by measured concentrations. The residuals scatter around zero without
a trend:

In [ ]:
fig, ax = plt.subplots(figsize=(5, 2.5))
bc.plot_residuals(results, ax=ax)
ax.set_xlabel("inhibitor (nM)")
ax.set_ylabel("residual (%)")
plt.show()

## Summarize

The summary combines the three independent experiments of each inhibitor. The
IC₅₀ is the geometric mean, with a 95% confidence interval:

In [ ]:
summary = results.summary()
summary[["compound_id", "N_fit", "IC50", "IC50_CI95_lower", "IC50_CI95_upper", "hill_slope"]]

In [ ]:
results.report(uncertainty="ci95", unit="nM")

The concentrations are in nM, so
$\mathrm{pIC}_{50} = 9 - \log_{10}(\mathrm{IC}_{50}/\mathrm{nM}) = 9 - \mathtt{logIC50}$.
The SD over independent experiments is the same on both log scales:

In [ ]:
summary["pIC50"] = 9 - summary["logIC50"]
summary[["compound_id", "pIC50", "logIC50_SD"]].round(2)

## Plot the summary

`plot_compounds` shows the mean of the three independent experiments at each
concentration, with the SD as error bars, and the curve at the summary
parameters:

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
bc.plot_compounds(results, ax=ax)
ax.set_xlabel("inhibitor (nM)")
ax.set_ylabel("activity (% of control)")
ax.legend()
plt.show()